# Clinical Score Comparison: SIRS & qSOFA on MIMIC-IV Test Set

Computes SIRS and qSOFA on the **same test patients** as the Multi-Agent model (E2) for a fair comparison. No retraining required.

**Steps:**
1. Load processed test data (vitals + labs)
2. Extract GCS from raw MIMIC-IV `chartevents.csv.gz` (test patients only)
3. Merge GCS into test set, forward-fill
4. Compute SIRS (4-criteria) and qSOFA (3-criteria) per row
5. Evaluate: sequence-level (last hour of each 24h window) and patient-level (max score)
6. Output comparison table + ROC curves

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
import gc
import os

PROJECT_PATH = '/content/drive/MyDrive/Sepsis'
DATA_PATH = f'{PROJECT_PATH}/data/processed/mimic_harmonized'
DATA_FILE = 'mimic_processed_full.h5'
RAW_PATH = '/content/drive/MyDrive/MIMIC'
OUTPUT_PATH = f'{PROJECT_PATH}/models/clinical_scores'
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEQ_LENGTH = 24
TEST_SIZE = 0.2
RANDOM_SEED = 42

# GCS item IDs in MIMIC-IV chartevents
GCS_ITEMIDS = [220739, 223900, 223901]  # eye, verbal, motor

print('Setup complete.')

## Step 1: Load processed data and reproduce test split

In [ ]:
# Load all features needed for SIRS + qSOFA
needed_cols = ['subject_id', 'charttime', 'sepsis_label',
               'resp', 'sbp', 'temp', 'hr', 'wbc', 'paco2']

print('Loading processed data...')
df = pd.read_hdf(f'{DATA_PATH}/{DATA_FILE}')
df = df[needed_cols].copy()
df['charttime'] = pd.to_datetime(df['charttime'])
print(f'Loaded {len(df):,} rows, {df["subject_id"].nunique():,} patients')

# Reproduce stratified split (same seed as training)
patient_ids = df['subject_id'].unique()
patient_labels = df.groupby('subject_id')['sepsis_label'].max().loc[patient_ids]

_, test_ids = train_test_split(
    patient_ids, test_size=TEST_SIZE,
    stratify=patient_labels, random_state=RANDOM_SEED
)

test_df = df[df['subject_id'].isin(test_ids)].copy()
print(f'Test patients: {len(test_ids):,}')
print(f'Test rows: {len(test_df):,}')

del df
gc.collect()

## Step 2: Extract GCS from raw chartevents (test patients only)

In [ ]:
# Read chartevents in chunks, keep only GCS rows for test patients
test_id_set = set(test_ids)
gcs_chunks = []

print('Reading chartevents.csv.gz (filtering for GCS + test patients)...')
chartevents_file = f'{RAW_PATH}/chartevents.csv.gz'

for i, chunk in enumerate(pd.read_csv(chartevents_file, chunksize=500_000,
                                       usecols=['subject_id', 'charttime', 'itemid', 'valuenum'],
                                       low_memory=False)):
    mask = chunk['itemid'].isin(GCS_ITEMIDS) & chunk['subject_id'].isin(test_id_set)
    filtered = chunk[mask]
    if len(filtered) > 0:
        gcs_chunks.append(filtered)
    if (i + 1) % 50 == 0:
        print(f'  Processed chunk {i+1}, found {sum(len(c) for c in gcs_chunks):,} GCS rows so far')

gcs_raw = pd.concat(gcs_chunks, ignore_index=True)
gcs_raw['charttime'] = pd.to_datetime(gcs_raw['charttime'])
print(f'\nTotal GCS rows extracted: {len(gcs_raw):,}')
print(gcs_raw['itemid'].value_counts())
del gcs_chunks
gc.collect()

## Step 3: Aggregate GCS components and merge into test set

In [ ]:
# Floor charttime to hour
gcs_raw['hour'] = gcs_raw['charttime'].dt.floor('h')

# Pivot: one column per GCS component
gcs_pivot = gcs_raw.pivot_table(
    index=['subject_id', 'hour'],
    columns='itemid',
    values='valuenum',
    aggfunc='last'
).reset_index()
gcs_pivot.columns.name = None
gcs_pivot = gcs_pivot.rename(columns={
    220739: 'gcs_eye',
    223900: 'gcs_verbal',
    223901: 'gcs_motor'
})

# Sum components -> total GCS
gcs_pivot['gcs'] = (gcs_pivot.get('gcs_eye', 0).fillna(0) +
                    gcs_pivot.get('gcs_verbal', 0).fillna(0) +
                    gcs_pivot.get('gcs_motor', 0).fillna(0))
# Where any component is missing, mark gcs as NaN (will be forward-filled)
all_components_present = gcs_pivot[['gcs_eye', 'gcs_verbal', 'gcs_motor']].notna().all(axis=1)
gcs_pivot.loc[~all_components_present, 'gcs'] = np.nan

print(f'GCS hourly records: {len(gcs_pivot):,}')
print(f'Patients with GCS: {gcs_pivot["subject_id"].nunique():,} / {len(test_ids):,}')
print(f'GCS distribution: min={gcs_pivot["gcs"].min()}, max={gcs_pivot["gcs"].max()}, mean={gcs_pivot["gcs"].mean():.1f}')

# Merge into test_df
test_df['hour'] = test_df['charttime'].dt.floor('h')
test_df = test_df.merge(
    gcs_pivot[['subject_id', 'hour', 'gcs']],
    on=['subject_id', 'hour'], how='left'
)

# Forward-fill GCS up to 12h per patient
test_df = test_df.sort_values(['subject_id', 'charttime'])
test_df['gcs'] = test_df.groupby('subject_id')['gcs'].ffill(limit=12)

print(f'\nTest rows with GCS: {test_df["gcs"].notna().sum():,} / {len(test_df):,} ({test_df["gcs"].notna().mean()*100:.1f}%)')

del gcs_raw, gcs_pivot
gc.collect()

## Step 4: Compute SIRS and qSOFA scores

In [ ]:
def compute_sirs(df):
    score = pd.Series(0, index=df.index)
    score += ((df['temp'] > 38) | (df['temp'] < 36)).fillna(False).astype(int)
    score += (df['hr'] > 90).fillna(False).astype(int)
    score += ((df['resp'] > 20) | (df['paco2'] < 32)).fillna(False).astype(int)
    score += ((df['wbc'] > 12) | (df['wbc'] < 4)).fillna(False).astype(int)
    return score

def compute_qsofa(df):
    score = pd.Series(0, index=df.index)
    score += (df['resp'] >= 22).fillna(False).astype(int)
    score += (df['sbp'] <= 100).fillna(False).astype(int)
    score += (df['gcs'] < 15).fillna(False).astype(int)
    return score

test_df['sirs_score'] = compute_sirs(test_df)
test_df['qsofa_score'] = compute_qsofa(test_df)

print('Score distributions:')
print('\nSIRS:')
print(test_df['sirs_score'].value_counts().sort_index())
print('\nqSOFA:')
print(test_df['qsofa_score'].value_counts().sort_index())

## Step 5: Sequence-level evaluation

For each 24h sequence (matching the model's evaluation), use **max score over the window** as the prediction. Label = `sepsis_label` of the last hour.

In [ ]:
# Build per-sequence aggregation matching model's eval
# Each sequence = 24 consecutive rows for a patient
# Label = last row's sepsis_label, prediction = max score over the window

test_df = test_df.sort_values(['subject_id', 'charttime']).reset_index(drop=True)

seq_records = []
for subject_id, group in test_df.groupby('subject_id', sort=False):
    n = len(group)
    if n < SEQ_LENGTH:
        continue
    sirs_arr = group['sirs_score'].values
    qsofa_arr = group['qsofa_score'].values
    label_arr = group['sepsis_label'].values
    for i in range(n - SEQ_LENGTH + 1):
        seq_records.append({
            'subject_id': subject_id,
            'sirs_max': sirs_arr[i:i+SEQ_LENGTH].max(),
            'qsofa_max': qsofa_arr[i:i+SEQ_LENGTH].max(),
            'label': label_arr[i+SEQ_LENGTH-1]
        })

seq_df = pd.DataFrame(seq_records)
del seq_records
gc.collect()
print(f'Total sequences: {len(seq_df):,}')
print(f'Positive rate: {seq_df["label"].mean()*100:.1f}%')

In [ ]:
# Sequence-level metrics
seq_results = {}
for name, col in [('SIRS', 'sirs_max'), ('qSOFA', 'qsofa_max')]:
    auroc = roc_auc_score(seq_df['label'], seq_df[col])
    auprc = average_precision_score(seq_df['label'], seq_df[col])
    seq_results[name] = {'auroc': auroc, 'auprc': auprc}
    print(f'{name}: AUROC={auroc:.4f}, AUPRC={auprc:.4f}')

## Step 6: Patient-level evaluation

In [ ]:
patient_df = seq_df.groupby('subject_id').agg(
    sirs_max=('sirs_max', 'max'),
    qsofa_max=('qsofa_max', 'max'),
    label=('label', 'max')
).reset_index()

print(f'Patients: {len(patient_df):,}')
print(f'Patient-level positive rate: {patient_df["label"].mean()*100:.1f}%')

patient_results = {}
for name, col in [('SIRS', 'sirs_max'), ('qSOFA', 'qsofa_max')]:
    auroc = roc_auc_score(patient_df['label'], patient_df[col])
    auprc = average_precision_score(patient_df['label'], patient_df[col])
    patient_results[name] = {'auroc': auroc, 'auprc': auprc}
    print(f'{name}: AUROC={auroc:.4f}, AUPRC={auprc:.4f}')

## Step 7: Comparison table

In [ ]:
MODEL_E2_SEQ = {'auroc': 0.7689, 'auprc': 0.7008}
MODEL_E2_PT = {'auroc': 0.8571, 'auprc': 0.7844}

print('='*70)
print('CLINICAL SCORE COMPARISON (same MIMIC-IV test set)')
print('='*70)
print(f'\n{"Model":<25}{"Seq AUROC":>12}{"Seq AUPRC":>12}{"Pt AUROC":>12}{"Pt AUPRC":>12}')
print('-'*73)
for name in ['SIRS', 'qSOFA']:
    s = seq_results[name]; p = patient_results[name]
    print(f'{name:<25}{s["auroc"]:>12.4f}{s["auprc"]:>12.4f}{p["auroc"]:>12.4f}{p["auprc"]:>12.4f}')
print(f'{"Multi-Agent E2":<25}{MODEL_E2_SEQ["auroc"]:>12.4f}{MODEL_E2_SEQ["auprc"]:>12.4f}{MODEL_E2_PT["auroc"]:>12.4f}{MODEL_E2_PT["auprc"]:>12.4f}')

print('\n\nMARKDOWN TABLE FOR THESIS:')
print('| Model | Seq AUROC | Seq AUPRC | Patient AUROC | Patient AUPRC |')
print('|-------|-----------|-----------|---------------|---------------|')
for name in ['SIRS', 'qSOFA']:
    s = seq_results[name]; p = patient_results[name]
    print(f'| {name} | {s["auroc"]:.4f} | {s["auprc"]:.4f} | {p["auroc"]:.4f} | {p["auprc"]:.4f} |')
print(f'| **Multi-Agent E2** | **{MODEL_E2_SEQ["auroc"]:.4f}** | **{MODEL_E2_SEQ["auprc"]:.4f}** | **{MODEL_E2_PT["auroc"]:.4f}** | **{MODEL_E2_PT["auprc"]:.4f}** |')

## Step 8: ROC curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sequence-level
ax = axes[0]
for name, col, color in [('SIRS', 'sirs_max', '#1f77b4'), ('qSOFA', 'qsofa_max', '#ff7f0e')]:
    fpr, tpr, _ = roc_curve(seq_df['label'], seq_df[col])
    ax.plot(fpr, tpr, label=f'{name} (AUROC={seq_results[name]["auroc"]:.3f})', color=color)
ax.plot([0,1], [0,1], 'k--', alpha=0.3)
ax.axhline(MODEL_E2_SEQ['auroc'], color='red', linestyle=':', alpha=0.5,
           label=f'Multi-Agent E2 AUROC={MODEL_E2_SEQ["auroc"]:.3f}')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Sequence-Level ROC')
ax.legend(); ax.grid(alpha=0.3)

# Patient-level
ax = axes[1]
for name, col, color in [('SIRS', 'sirs_max', '#1f77b4'), ('qSOFA', 'qsofa_max', '#ff7f0e')]:
    fpr, tpr, _ = roc_curve(patient_df['label'], patient_df[col])
    ax.plot(fpr, tpr, label=f'{name} (AUROC={patient_results[name]["auroc"]:.3f})', color=color)
ax.plot([0,1], [0,1], 'k--', alpha=0.3)
ax.axhline(MODEL_E2_PT['auroc'], color='red', linestyle=':', alpha=0.5,
           label=f'Multi-Agent E2 AUROC={MODEL_E2_PT["auroc"]:.3f}')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Patient-Level ROC')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/clinical_score_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {OUTPUT_PATH}/clinical_score_roc.png')

In [ ]:
# Save results to disk for thesis use
import json
results_out = {
    'sequence_level': seq_results,
    'patient_level': patient_results,
    'n_sequences': int(len(seq_df)),
    'n_patients': int(len(patient_df)),
    'sequence_prevalence': float(seq_df['label'].mean()),
    'patient_prevalence': float(patient_df['label'].mean())
}
with open(f'{OUTPUT_PATH}/clinical_score_results.json', 'w') as f:
    json.dump(results_out, f, indent=2)
print('Results saved.')